# Kesehatan: HL7 v2 dan DICOM

Monitor pasien, analyzer, dan modalitas berbicara **HL7 v2** (kejadian dan hasil lewat MLLP) dan **DICOM** (gambar lewat C-STORE). Semua di sini sintetis — pasien fiktif dan phantom skematis, tidak untuk penggunaan klinis.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.15.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.15.0-preview.1"
#r "nuget: IoTCom.Net.Adapters.Dicom, 0.15.0-preview.1"

## Bangun dan urai ORU^R01
Satu segmen OBX per pengukuran, berkode LOINC.

In [ ]:
using IoTCom.Net.Protocols.Hl7;

var (patient, scenario, bed) = PatientMonitorSimulator.DemoWard[0];
var monitor = new PatientMonitorSimulator(patient, scenario, bed, onset: TimeSpan.Zero);
var oru = monitor.ToOru(monitor.Next(TimeSpan.FromMinutes(8), DateTimeOffset.Now));
Console.WriteLine(oru.Encode().Replace('\r', '\n'));
foreach (var o in Hl7Message.Parse(oru.Encode()).GetObservations())
    Console.WriteLine($"{o.Code.Text,-36} {o.Value,6} {o.Units,-8} {o.AbnormalFlag}");

## Kirim lewat MLLP dan terima ACK

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Transports;

var link = new InMemoryTransportListener();
await using var receiver = Hl7MllpServer.Create(o => o.ListenInMemory(link));
receiver.MessageReceived += (_, e) => Console.WriteLine($"received {e.Message.MessageType} {e.Message.ControlId}");
await receiver.StartAsync();
await using var sender = Hl7MllpClient.Create(o => o.UseInMemory(link));
var ack = await sender.SendAsync(oru);
Console.WriteLine($"ACK {ack.AckCode()} for {ack["MSA.2"]}");

## DICOM: studi sintetis lewat C-STORE, dirender dengan window
Temuan yang ditanam adalah ground truth (berguna untuk menguji alur AI).

In [ ]:
using IoTCom.Net.Adapters.Dicom;

await using var pacs = DicomStoreServer.Create(o => o.Port = 11120);
await pacs.StartAsync();
DicomReceived? got = null;
pacs.ImageReceived += r => got = r;
await using var modality = DicomStoreClient.Create(o => o.Port = 11120);
await modality.EchoAsync();
var study = SyntheticImaging.Generate(SyntheticModality.ChestCt, SyntheticFinding.Pneumothorax);
await modality.StoreAsync(study.File);
var image = DicomRenderer.Render(got!.File.Dataset, DicomRenderer.Presets["CT lung"]);
Console.WriteLine($"{got.Modality} {image.Width}x{image.Height}, PNG {image.ToPng().Length / 1024} KB — planted: {study.FindingDescription}");

## Lebih lanjut
Demo Galeri *Monitor pasien ICU* dan *Pra-baca gambar dengan AI* menambahkan skor NEWS2, analisis tren, dan LLM (catatan SBAR dan pra-baca vision) di atas blok-blok ini — lihat `docs/id/guides/medical-ai.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*